<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/remoteclip-ucm/blob/main/02_remoteclip_zero_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset

ds = load_dataset("rifahnanjiba2/UC_Merced")

README.md:   0%|          | 0.00/2.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  416MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2100 [00:00<?, ? examples/s]

In [2]:
!pip install -q open_clip_torch huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00


In [3]:
import torch
import open_clip

from datasets import load_dataset, DatasetDict
from huggingface_hub import hf_hub_download


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


# Split the dataset

In [4]:
data = ds["train"]

# 70% train, 30% temporary
split_1 = data.train_test_split(
    test_size=0.30,
    seed=42,
    stratify_by_column="label"
)

# 15% validation, 15% test
split_2 = split_1["test"].train_test_split(
    test_size=0.50,
    seed=42,
    stratify_by_column="label"
)

ds = DatasetDict({
    "train": split_1["train"],
    "validation": split_2["train"],
    "test": split_2["test"]
})

print(ds)


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1470
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 315
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 315
    })
})


In [5]:
class_names = ds["train"].features["label"].names

print("Number of classes:", len(class_names))
print(class_names)


Number of classes: 21
['agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings', 'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse', 'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass', 'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt']


# Load RemoteCLIP

In [8]:
checkpoint_path = hf_hub_download(
    repo_id="chendelong/RemoteCLIP",
    filename="RemoteCLIP-ViT-B-32.pt"
)

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained=None
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

tokenizer = open_clip.get_tokenizer("ViT-B-32")

print("RemoteCLIP loaded on:", device)


RemoteCLIP loaded on: cuda


# Create 21 text prompts

In [9]:
text_queries = [
    "A satellite image of agricultural land.",
    "A satellite image of an airplane.",
    "A satellite image of a baseball diamond.",
    "A satellite image of a beach.",
    "A satellite image of buildings.",
    "A satellite image of chaparral vegetation.",
    "A satellite image of dense residential buildings.",
    "A satellite image of a forest.",
    "A satellite image of a freeway.",
    "A satellite image of a golf course.",
    "A satellite image of a harbor.",
    "A satellite image of an intersection.",
    "A satellite image of medium-density residential buildings.",
    "A satellite image of a mobile home park.",
    "A satellite image of an overpass.",
    "A satellite image of a parking lot.",
    "A satellite image of a river.",
    "A satellite image of a runway.",
    "A satellite image of sparse residential buildings.",
    "A satellite image of storage tanks.",
    "A satellite image of a tennis court."
]

print("Number of prompts:", len(text_queries))


Number of prompts: 21


In [10]:
text = tokenizer(text_queries).to(device)

with torch.no_grad():
    text_features = model.encode_text(text)
    text_features /= text_features.norm(
        dim=-1,
        keepdim=True
    )

print("Text features:", text_features.shape)


Text features: torch.Size([21, 512])


In [13]:
sample = ds["test"][0]

image = sample["image"]
true_label = sample["label"]

# Preprocess image
image_input = preprocess(image).unsqueeze(0).to(device)

# Get image features and probabilities
with torch.no_grad():
    image_features = model.encode_image(image_input)

    image_features /= image_features.norm(
        dim=-1,
        keepdim=True
    )

    # Calculate similarity with all text classes
    similarity = 100.0 * image_features @ text_features.T

    # Convert similarities to probabilities
    probabilities = similarity.softmax(dim=-1)

# Get predicted class
predicted_label = probabilities.argmax(dim=-1).item()

# Print all predictions
print("Predictions of RN50:")

for class_name, probability in zip(
    class_names,
    probabilities[0]
):
    print(
        f"{class_name:<45} "
        f"{probability.item() * 100:.1f}%"
    )

# Print actual and predicted classes
print()
print("True class:", class_names[true_label])
print("Predicted class:", class_names[predicted_label])

# Print confidence
print(
    "Confidence:",
    f"{probabilities[0, predicted_label].item() * 100:.2f}%"
)


Predictions of RN50:
agricultural                                  0.0%
airplane                                      0.0%
baseballdiamond                               0.0%
beach                                         0.0%
buildings                                     0.0%
chaparral                                     100.0%
denseresidential                              0.0%
forest                                        0.0%
freeway                                       0.0%
golfcourse                                    0.0%
harbor                                        0.0%
intersection                                  0.0%
mediumresidential                             0.0%
mobilehomepark                                0.0%
overpass                                      0.0%
parkinglot                                    0.0%
river                                         0.0%
runway                                        0.0%
sparseresidential                             0.0%
storaget

# Zero-shot evaluation

In [14]:
correct = 0
total = 0

all_predictions = []
all_labels = []

for i, sample in enumerate(ds["test"]):

    image = sample["image"]
    label = sample["label"]

    image_input = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_input)

        image_features /= image_features.norm(
            dim=-1,
            keepdim=True
        )

        probabilities = (
            100.0 * image_features @ text_features.T
        ).softmax(dim=-1)

        prediction = probabilities.argmax(dim=-1).item()

    all_predictions.append(prediction)
    all_labels.append(label)

    if prediction == label:
        correct += 1

    total += 1

    # Print predictions for this image
    print(f"\nImage {i + 1}")
    print("Predictions of RN50:")

    for class_name, probability in zip(
        class_names,
        probabilities[0]
    ):
        print(
            f"{class_name:<45} "
            f"{probability.item():.1f}%"
        )

    print()
    print("True class:", class_names[label])
    print("Predicted class:", class_names[prediction])
    print(
        "Confidence:",
        f"{probabilities[0, prediction].item():.2f}%"
    )

accuracy = correct / total

print("\n" + "=" * 50)
print(f"Zero-shot accuracy: {accuracy * 100:.2f}%")


Streaming output truncated to the last 5000 lines.
mediumresidential                             0.0%
mobilehomepark                                0.9%
overpass                                      0.0%
parkinglot                                    0.0%
river                                         0.0%
runway                                        0.0%
sparseresidential                             0.0%
storagetanks                                  0.0%
tenniscourt                                   0.0%

True class: denseresidential
Predicted class: mobilehomepark
Confidence: 0.94%

Image 138
Predictions of RN50:
agricultural                                  0.0%
airplane                                      0.0%
baseballdiamond                               0.0%
beach                                         0.0%
buildings                                     0.0%
chaparral                                     0.0%
denseresidential                              0.0%
forest               